In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
# We will only keep the 10,000 most frequent words in the training data to avoid weird typos
vocab_size = 10000

print("Loading data...")
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)

print(f"Training entries: {len(X_train)}, Labels: {len(y_train)}")
print(f"Example review (in numbers): {X_train[0][:10]}...")

Loading data...
17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Training entries: 25000, Labels: 25000
Example review (in numbers): [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65]...


In [3]:
max_length = 256

X_train = pad_sequences(X_train, maxlen=max_length, padding='post', value=0)
X_test = pad_sequences(X_test, maxlen=max_length, padding='post', value=0)

print(f"New shape of training data: {X_train.shape}")

New shape of training data: (25000, 256)


In [4]:
model = Sequential()

# 1. The Embedding Layer: Turns integer words into dense vectors of meaning
model.add(Embedding(vocab_size, 16))

# 2. Global Average Pooling: Squashes the sequence down so the Dense layer can read it
model.add(GlobalAveragePooling1D())

# 3. Hidden Dense Layer
model.add(Dense(16, activation='relu'))

# 4. Output Layer
model.add(Dense(1, activation='sigmoid'))

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling1d             │ ?                           │               0 │
│ (GlobalAveragePooling1D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [5]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [6]:
history = model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=512,
    validation_split=0.2
)

Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 7s 65ms/step - accuracy: 0.6094 - loss: 0.6879 - val_accuracy: 0.7122 - val_loss: 0.6781
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.7148 - loss: 0.6642 - val_accuracy: 0.7400 - val_loss: 0.6429
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.7524 - loss: 0.6179 - val_accuracy: 0.7836 - val_loss: 0.5855
Epoch 4/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.7986 - loss: 0.5532 - val_accuracy: 0.8128 - val_loss: 0.5194
Epoch 5/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 39ms/step - accuracy: 0.8259 - loss: 0.4858 - val_accuracy: 0.8124 - val_loss: 0.4643
Epoch 6/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 40ms/step - accuracy: 0.8399 - loss: 0.4315 - val_accuracy: 0.8450 - val_loss: 0.4175
Epoch 7/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 3s 47ms/step - accuracy: 0.8620 - loss: 0.3845 - val_accuracy: 0.8124 - val_loss: 0.4090
Epoch 8/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 42ms/step - accuracy: 0.8684 - loss: 0.3549 - val_accuracy: 0.8398 - v

In [7]:
loss, accuracy = model.evaluate(X_test,  y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")

# Let's see what it predicts for the first test review
prediction = model.predict(X_test[:1])
sentiment = "Positive" if prediction[0][0] > 0.5 else "Negative"

print(f"Model Prediction: {sentiment} ({prediction[0][0]:.4f})")
print(f"Actual Label: {'Positive' if y_test[0] == 1 else 'Negative'}")

782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.8820 - loss: 0.2902
Accuracy: 88.20%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step
Model Prediction: Negative (0.1602)
Actual Label: Negative
